# Лабораторная работа: классификация ЭКГ. Часть 2 — классификатор

В этой части вы обучите нейросетевой классификатор диагнозов на датасете, который собрали и экспортировали в первой части (`ecg-warehouse/notebooks/lab_dataset.ipynb`), оцените его на тестовой выборке и попробуете улучшить базовую модель.

Обучение построено на PyTorch Lightning и библиотеке `ecg_ml`: `WarehouseDatasetModule` загружает экспорт `ecg-warehouse`, `ClassifierLModule` реализует шаги обучения, валидации и теста.

Задача **многометочная** (multi-label): классы не взаимоисключающие, у одной записи может быть несколько кодов варианта сразу (например, `IMI` и `ASMI`) или ни одного. Поэтому у сети по одному независимому выходу на класс, а не softmax по классам.

Ноутбук выполняется в окружении проекта `ecg-ml`: установите его командой `uv sync` в корне репозитория и выберите ядро из `.venv`.

In [ ]:
import os
from pathlib import Path

# Пути в ноутбуке отсчитываются от корня репозитория, а Jupyter может запустить ноутбук из notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir("..")

### Параметры

- `VARIANT` — номер вашего варианта, тот же, что в первой части;
- `DATASET_PATH` — каталог, в который первая часть экспортировала датасет (путь относительно корня репозитория `ecg-ml`);
- `LEARNING_RATE`, `BATCH_SIZE`, `EPOCHS` — шаг обучения, размер батча и максимальное число эпох;
- `NUM_WORKERS` — число процессов, которые готовят батчи.

In [ ]:
VARIANT = 1
DATASET_PATH = f"data/ptb-xl-variant-{VARIANT}"

LEARNING_RATE = 0.1
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS = 30

In [ ]:
from functools import partial
import torch
from lightning import Trainer
from ecg_ml.data.warehouse import WarehouseDatasetModule
from ecg_ml.modules.classifier import ClassifierLModule

## 1. Данные

`WarehouseDatasetModule` читает `signals.npz` и `meta.csv` из экспорта. Тестовая выборка — часть `test` из разбиения хранилища. Валидационная выделяется из `train` тоже по пациентам, около 10 % пациентов; `random_state` фиксирует это разбиение, чтобы модели сравнивались на одних и тех же записях.

Классы и их порядок берутся из экспорта: `CLASSES` совпадёт со списком из первой части.

In [ ]:
dataset = WarehouseDatasetModule(DATASET_PATH, random_state=0)
datamodule = dataset.to_datamodule(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

CLASSES = dataset.classes
CLASSES

## 2. Модель

В ячейке ниже — заготовка `ECGClassifier`.

**Задание.** Замените реализацию `ECGClassifier` в ячейке ниже собственной архитектурой нейросети для классификации ЭКГ. Сохраните имя класса и параметр `output_size`: модель создаётся в ячейке с `ClassifierLModule` как `ECGClassifier(output_size=len(CLASSES))`. Требования к модели:

- вход — батч сигналов формы `(batch, 12, 5000)`: 12 отведений по 5000 отсчётов;
- выход — логиты формы `(batch, len(CLASSES))`.

In [ ]:
class ECGClassifier(torch.nn.Module):
    def __init__(self, output_size=2):
        super().__init__()
        self.flatten = torch.nn.Flatten()
        self.linear = torch.nn.Linear(12 * 5000, output_size)

    def forward(self, x):
        x = self.flatten(x)
        x = self.linear(x)

        return x

### Функция потерь и оптимизатор

- `BCEWithLogitsLoss` — бинарная кросс-энтропия, посчитанная независимо для каждого класса; подходит для многометочной задачи;
- `Adam` с шагом обучения `LEARNING_RATE`;
- `ReduceLROnPlateau` уменьшает шаг в 10 раз, если `val/loss` не улучшается 10 эпох подряд.

In [ ]:
optimizer = partial(torch.optim.Adam, lr=LEARNING_RATE)
lr_scheduler = {
    "scheduler": partial(
        torch.optim.lr_scheduler.ReduceLROnPlateau, mode="min", factor=0.1, patience=10
    ),
    "monitor": "val/loss",
    "interval": "epoch",
    "frequency": 1,
}
criterion = torch.nn.BCEWithLogitsLoss()

`ClassifierLModule` объединяет сеть, функцию потерь, оптимизатор и метрики. На валидации и тесте считаются precision, recall и F1 для каждого класса и их macro-среднее.

In [ ]:
classifier = ClassifierLModule(
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    criterion=criterion,
    network=ECGClassifier(output_size=len(CLASSES)),
    labels=CLASSES,
)

## 3. Обучение

`Trainer` проводит валидацию после каждой эпохи и обучает не больше `EPOCHS` эпох в смешанной точности (`precision=16`). Метрики по эпохам сохраняются в `logs/lab/lightning_logs/version_N/metrics.csv`, веса — в `logs/lab/lightning_logs/version_N/checkpoints/`. Каталог `logs/` исключён из git.

In [ ]:
trainer = Trainer(
    precision=16,
    check_val_every_n_epoch=1,
    min_epochs=1,
    max_epochs=EPOCHS,
    default_root_dir="logs/lab",
)

In [ ]:
trainer.fit(
    model=classifier,
    datamodule=datamodule,
)

## 4. Оценка на тестовой выборке

`trainer.test` считает метрики на test для весов после последней эпохи и печатает `classification_report` из scikit-learn: precision, recall и F1 по каждому классу. Тестовую выборку используйте только для итоговой оценки, а модели и гиперпараметры сравнивайте по валидации.

In [ ]:
trainer.test(
    model=classifier,
    datamodule=datamodule,
)

## 5. Улучшение модели

Что можно попробовать для улучшения:

- подобрать `LEARNING_RATE`, `BATCH_SIZE`, `EPOCHS`;
- учесть дисбаланс классов: `torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(dataset.pos_weight))`;
- изменить архитектуру `ECGClassifier`: число и ширину слоёв, размер ядер свёрток, нормализацию, dropout.

## Сдача работы

Сохраните ноутбук вместе с результатами ячеек, закоммитьте его в свою ветку и откройте запросы на слияние в обоих форках, как описано в инструкции к лабораторной работе.